In [15]:
import numpy as np
import time
from numba import njit
from scipy.spatial.distance import cdist

# Fixed transitions and weights
# Corresponds to steps = [[1,1], [0,1], [1,0]]
# and weights = [2, 1, 1]
step_r = np.array([1, 0, 1])
step_c = np.array([1, 1, 0])
weights = np.array([2.0, 1.0, 1.0], dtype=np.float32)
step_transitions = np.array([(1 << 1) | 1, (0 << 1) | 1, (1 << 1) | 0], dtype=np.int8)

# Lookup tables for B decoding
# These lookup tables are used to quickly decode the transition type and repetition count 
# from the backtracking matrix `B`, where each entry is a compact 8-bit integer:
# 
# Bits 0–1 (2 bits): Encodes the step direction (i.e., which step was taken — diagonal, right, or down).
# Bits 2–7 (6 bits): Encodes how many times this same step direction has been taken consecutively.
#
# These tables allow us to avoid doing bitwise operations during backtracking, which speeds things up.

# For each possible 8-bit value (0 to 255), extract the lower 2 bits (bits 0 and 1),
# which represent the step direction (transition type)


bit_transition_lookup = np.array([b & 0b11 for b in range(256)], dtype=np.uint8)
bit_count_lookup = np.array([b >> 2 for b in range(256)], dtype=np.uint8)

@njit
def compute_cost_matrix(F1, F2, downsample):
    F1_sub = np.ascontiguousarray(F1[:, 0::downsample].T) # Ensures the data is stored in a memory layout that Numba likes (C-contiguous).
    F2_sub = np.ascontiguousarray(F2[:, 0::downsample])
    return 1 - F1_sub @ F2_sub

@njit
def compute_D_and_B(C, hyperparam):
    rows, cols = C.shape
    D = np.full((rows, cols), np.inf)
    B = np.zeros((rows, cols), dtype=np.uint8)
    D[0, 0] = C[0, 0] # Ensures we force the first frames to match, avoiding subsequence DTW
    
    for row in range(rows):
        for col in range(cols):
            if (row ==0 and col==0): 
                pass
            else:
                mincost = np.inf
                minidx = -1
    
                for i in range(3):  # 3 fixed steps, apparently numba doesn't like enumerate so we hardcoded the range
                    prevrow = row - step_r[i]
                    prevcol = col - step_c[i]
                    if prevrow >= 0 and prevcol >= 0: # Check if step moves within boundaries
                        cur_transition = step_transitions[i]
                        prev_b = B[prevrow, prevcol]
                        prev_transition = bit_transition_lookup[prev_b]
                        prev_count = bit_count_lookup[prev_b]
    
                        pathcost = D[prevrow, prevcol] + C[row, col] * weights[i]
    
                        if cur_transition == 0b11: # If current step is diaganol
                            if pathcost < mincost:
                                mincost = pathcost
                                minidx = (0 << 2) | cur_transition
                        elif prev_transition == cur_transition: # If previous step is repeated
                            if prev_count < hyperparam and pathcost < mincost:
                                mincost = pathcost
                                minidx = ((prev_count + 1) << 2) | cur_transition
                        else: # Switching from right to up step or vice versa
                            if pathcost < mincost:
                                mincost = pathcost
                                minidx = (1 << 2) | cur_transition
    
                D[row, col] = mincost
                B[row, col] = minidx

    return D, B

@njit
def backtrace_MDTW_Numba(D, B):
    endcol = D.shape[1] -1
    row = D.shape[0] - 1
    col = endcol

    rowpath = [row]
    colpath = [col]

    while not(row <= 0 and col<=0):
        bInt = B[row, col]
        rstep = (bInt >> 1) & 1
        cstep = bInt & 1
        row -= rstep
        col -= cstep
        rowpath.append(row)
        colpath.append(col)

    path = np.empty((2, len(rowpath)), dtype=np.int64)
    for i in range(len(rowpath)):
        path[0, i] = rowpath[len(rowpath) - 1 - i]
        path[1, i] = colpath[len(colpath) - 1 - i]

    return path

def alignMDTW_Numba(featfile1, featfile2, hyperparam, downsample, outfile=None, profile=False):
    
    F1data = np.load(featfile1) # 88 x N
    F1 = F1data['roll']
    F2data = np.load(featfile2) # 88 x M
    F2 = F2data['roll']

    print(F1.shape)
    print(F2.shape)

    if max(F1.shape[1], F2.shape[1]) / min(F1.shape[1], F2.shape[1]) >= hyperparam + 1:
        if outfile is not None:
            import pickle
            # Creates a file with only None
            pickle.dump(None, open(outfile, 'wb'))
        return None

    times = []
    times.append(time.time())
    
    # jaccard distance
    F1_sampled = F1[:,::downsample].T
    F2_sampled = F2[:,::downsample].T
    C = cdist(F1_sampled, F2_sampled, metric='jaccard')

    times.append(time.time())
    D, B = compute_D_and_B(C, hyperparam)
    times.append(time.time())
    path = backtrace_MDTW_Numba(D, B)
    times.append(time.time())

    if outfile is not None:
        import pickle
        pickle.dump(path, open(outfile, 'wb'))

    if profile:
        return path, np.diff(times)
    else:
        return path, C

Align a single pair of audio files with Numba Modified DTW

In [17]:
# featfile1 = 'midi_features/schumann_arabeske_park.mid.npz'
# featfile2 = 'xml_features/schumann_arabeske.musicxml.npz'
# hyperparam = 2
# downsample = 1
# steps = np.array([1,0,0,1,1,1]).reshape((-1,2))
# weights = np.array([1,1,2])
# wp2 = alignMDTW_Numba(featfile1, featfile2, hyperparam, downsample)

(88, 7873)
(88, 7608)
